In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

Importando Bibliotecas 

In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.base import clone

from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier
)

from sklearn.metrics import (
    accuracy_score,
    recall_score,
    classification_report
)

Carregando Dataset 

In [2]:
df = pd.read_csv("/kaggle/input/datasets/menegidio/iris-species/Iris.csv")

df.head()

,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,1,5.1,3.5,1.4,0.2,Iris-setosa
1,2,4.9,3.0,1.4,0.2,Iris-setosa
2,3,4.7,3.2,1.3,0.2,Iris-setosa
3,4,4.6,3.1,1.5,0.2,Iris-setosa
4,5,5.0,3.6,1.4,0.2,Iris-setosa


Preparando dados 

In [3]:
X = df.drop(columns=["Id", "Species"])

y = df["Species"]

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Treinamento:", X_treino.shape)
print("Teste:", X_teste.shape)

Treinamento: (120, 4)
Teste: (30, 4)


Criação dos modelos sem GrindSearchCV

No svm utilizei um pipeline para padronizar os dados antes do treino.

In [6]:
modelos = {
    "SVM": Pipeline([
        ("padronizacao", StandardScaler()),
        ("svm", SVC())
    ]),

    "Árvore de Decisão": DecisionTreeClassifier(
        random_state=42
    ),

    "Floresta Aleatória": RandomForestClassifier(
        random_state=42
    ),

    "Boosting": GradientBoostingClassifier(
        random_state=42
    )
}

Treinamento sem Grid

In [8]:
resultados = []

for nome, modelo in modelos.items():
    # Treinamento
    modelo.fit(X_treino, y_treino)

    # Previsões
    previsoes = modelo.predict(X_teste)

    # Cálculo das métricas
    acuracia = accuracy_score(y_teste, previsoes)

    sensibilidade = recall_score(
        y_teste,
        previsoes,
        average="macro"
    )

    resultados.append({
        "Modelo": nome,
        "Versão": "Sem GridSearchCV",
        "Acurácia": acuracia,
        "Sensibilidade": sensibilidade,
        "Melhores parâmetros": "Parâmetros padrão"
    })

Resultado sem Grid

In [9]:
resultado_sem_grid = pd.DataFrame(resultados)

resultado_sem_grid

,Modelo,Versão,Acurácia,Sensibilidade,Melhores parâmetros
0,SVM,Sem GridSearchCV,0.966667,0.966667,Parâmetros padrão
1,Árvore de Decisão,Sem GridSearchCV,0.933333,0.933333,Parâmetros padrão
2,Floresta Aleatória,Sem GridSearchCV,0.900000,0.900000,Parâmetros padrão
3,Boosting,Sem GridSearchCV,0.966667,0.966667,Parâmetros padrão


Mostrando metricas como porcentagem

In [10]:
resultado_formatado = resultado_sem_grid.copy()

resultado_formatado["Acurácia"] = (
    resultado_formatado["Acurácia"] * 100
).round(2)

resultado_formatado["Sensibilidade"] = (
    resultado_formatado["Sensibilidade"] * 100
).round(2)

resultado_formatado

,Modelo,Versão,Acurácia,Sensibilidade,Melhores parâmetros
0,SVM,Sem GridSearchCV,96.67,96.67,Parâmetros padrão
1,Árvore de Decisão,Sem GridSearchCV,93.33,93.33,Parâmetros padrão
2,Floresta Aleatória,Sem GridSearchCV,90.00,90.00,Parâmetros padrão
3,Boosting,Sem GridSearchCV,96.67,96.67,Parâmetros padrão


Config do Grind 

In [12]:
parametros = {
    "SVM": {
        "svm__C": [0.1, 1, 10, 100],
        "svm__kernel": ["linear", "rbf"],
        "svm__gamma": ["scale", "auto"]
    },

    "Árvore de Decisão": {
        "criterion": ["gini", "entropy"],
        "max_depth": [None, 2, 3, 4, 5, 10],
        "min_samples_split": [2, 5, 10]
    },

    "Floresta Aleatória": {
        "n_estimators": [50, 100, 200],
        "max_depth": [None, 3, 5, 10],
        "min_samples_split": [2, 5]
    },

    "Boosting": {
        "n_estimators": [50, 100, 200],
        "learning_rate": [0.05, 0.1, 0.2],
        "max_depth": [1, 2, 3]
    }
}

Treinando com Grind 

In [13]:
melhores_modelos = {}

for nome, modelo in modelos.items():
    print("Executando GridSearchCV para:", nome)

    grid = GridSearchCV(
        estimator=clone(modelo),
        param_grid=parametros[nome],
        scoring={
            "acuracia": "accuracy",
            "sensibilidade": "recall_macro"
        },
        refit="acuracia",
        cv=5,
        n_jobs=-1
    )


    grid.fit(X_treino, y_treino)

    melhor_modelo = grid.best_estimator_
    melhores_modelos[nome] = melhor_modelo

    previsoes = melhor_modelo.predict(X_teste)

   
    acuracia = accuracy_score(y_teste, previsoes)

    sensibilidade = recall_score(
        y_teste,
        previsoes,
        average="macro"
    )

    resultados.append({
        "Modelo": nome,
        "Versão": "Com GridSearchCV",
        "Acurácia": acuracia,
        "Sensibilidade": sensibilidade,
        "Melhores parâmetros": grid.best_params_
    })

Executando GridSearchCV para: SVM
Executando GridSearchCV para: Árvore de Decisão
Executando GridSearchCV para: Floresta Aleatória
Executando GridSearchCV para: Boosting


Comparação geral

In [17]:
comparacao = pd.DataFrame(resultados)

comparacao = comparacao.sort_values(
    by=["Acurácia", "Sensibilidade"],
    ascending=False
)

comparacao

,Modelo,Versão,Acurácia,Sensibilidade,Melhores parâmetros
0,SVM,Sem GridSearchCV,0.966667,0.966667,Parâmetros padrão
3,Boosting,Sem GridSearchCV,0.966667,0.966667,Parâmetros padrão
6,Floresta Aleatória,Com GridSearchCV,0.966667,0.966667,"{'max_depth': 3, 'min_samples_split': 2, 'n_es..."
7,Boosting,Com GridSearchCV,0.966667,0.966667,"{'learning_rate': 0.1, 'max_depth': 3, 'n_esti..."
1,Árvore de Decisão,Sem GridSearchCV,0.933333,0.933333,Parâmetros padrão
4,SVM,Com GridSearchCV,0.933333,0.933333,"{'svm__C': 0.1, 'svm__gamma': 'scale', 'svm__k..."
5,Árvore de Decisão,Com GridSearchCV,0.933333,0.933333,"{'criterion': 'gini', 'max_depth': None, 'min_..."
2,Floresta Aleatória,Sem GridSearchCV,0.900000,0.900000,Parâmetros padrão


In [18]:
comparacao_formatada = comparacao.copy()

comparacao_formatada["Acurácia"] = (
    comparacao_formatada["Acurácia"] * 100
).round(2)

comparacao_formatada["Sensibilidade"] = (
    comparacao_formatada["Sensibilidade"] * 100
).round(2)

comparacao_formatada

,Modelo,Versão,Acurácia,Sensibilidade,Melhores parâmetros
0,SVM,Sem GridSearchCV,96.67,96.67,Parâmetros padrão
3,Boosting,Sem GridSearchCV,96.67,96.67,Parâmetros padrão
6,Floresta Aleatória,Com GridSearchCV,96.67,96.67,"{'max_depth': 3, 'min_samples_split': 2, 'n_es..."
7,Boosting,Com GridSearchCV,96.67,96.67,"{'learning_rate': 0.1, 'max_depth': 3, 'n_esti..."
1,Árvore de Decisão,Sem GridSearchCV,93.33,93.33,Parâmetros padrão
4,SVM,Com GridSearchCV,93.33,93.33,"{'svm__C': 0.1, 'svm__gamma': 'scale', 'svm__k..."
5,Árvore de Decisão,Com GridSearchCV,93.33,93.33,"{'criterion': 'gini', 'max_depth': None, 'min_..."
2,Floresta Aleatória,Sem GridSearchCV,90.00,90.00,Parâmetros padrão


Indentificação do melhor modelo (automatica)

In [19]:
melhor_resultado = comparacao.iloc[0]

print("Melhor modelo:", melhor_resultado["Modelo"])
print("Versão:", melhor_resultado["Versão"])

print(
    f"Acurácia: "
    f"{melhor_resultado['Acurácia']:.2%}"
)

print(
    f"Sensibilidade: "
    f"{melhor_resultado['Sensibilidade']:.2%}"
)

print(
    "Parâmetros:",
    melhor_resultado["Melhores parâmetros"]
)

Melhor modelo: SVM
Versão: Sem GridSearchCV
Acurácia: 96.67%
Sensibilidade: 96.67%
Parâmetros: Parâmetros padrão
